# TIPSv2 — Spatial Features, Zero-Shot Segmentation & Dense Prediction

**TIPSv2** (Text-Image Pre-training with Spatial awareness) is Google's contrastive
vision-language model. Unlike CLIP-style models that produce only a single global
embedding, TIPSv2 outputs **dense per-patch features aligned with text** — enabling
spatial understanding without any task-specific fine-tuning.

This notebook uses the **`-dpt` variant**, which bundles the TIPS backbone with three
frozen-backbone DPT (Dense Prediction Transformer) heads trained on top of it:

| Part | Capability | Model component |
|---|---|---|
| **1** | PCA & K-means feature visualisation | TIPS vision encoder |
| **2** | Zero-shot text-guided segmentation | TIPS vision + text encoder |
| **3** | Monocular depth & surface normals | DPT head (NYU Depth V2) |
| **4** | Supervised semantic segmentation | DPT head (ADE20K, 150 classes) |

All heavy functions live in `tipsv2_seg_utils.py`.

**Prerequisites:** `apple_pear_1.jpg` scene image, Fruits360 dataset,
`transformers`, `torch`, `torchvision`, `sentencepiece`, `scikit-learn`.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import sys; sys.path.insert(0, ".")

In [ ]:
%cd ~/cvi4ic-notebooks/12
from tipsv2_seg_utils import (
    DEFAULT_RESOLUTION,
    load_model, extract_features, extract_features_value_attention,
    encode_text_classes, zeroseg,
    vis_pca, vis_depth_pca, vis_kmeans,
    run_depth, run_normals, run_segmentation,
    vis_depth_dpt, vis_normals_dpt, vis_segmentation_dpt,
    show_pca_panel, show_pca_gallery, show_kmeans_sweep,
    show_zeroseg, show_zeroseg_comparison,
    show_depth_normals, show_supervised_seg,
    get_image_paths, find_class,
)
print("Imports OK.")

In [ ]:
# ── Edit these two paths ──────────────────────────────────────────────
DATASET_PATH = pathlib.Path(r"../fruits-360-100x100")
SCENE_PATH   = pathlib.Path(r"apple_pear_1.jpg")
# ─────────────────────────────────────────────────────────────────────

TRAIN_ROOT = DATASET_PATH / "Training"
assert TRAIN_ROOT.exists(), f"Training/ not found under {DATASET_PATH}"
assert SCENE_PATH.exists(), f"Scene image not found: {SCENE_PATH}"

classes   = sorted([d.name for d in TRAIN_ROOT.iterdir() if d.is_dir()])
scene_arr = np.array(Image.open(SCENE_PATH).convert("RGB"))
orig_h, orig_w = scene_arr.shape[:2]

print(f"Dataset : {len(classes)} classes")
print(f"Scene   : {SCENE_PATH.name}  {orig_w} × {orig_h} px")

## Authenticate with Hugging Face

The TIPSv2-DPT weights are hosted on Hugging Face.

1. Create an account at huggingface.co
2. **Settings → Access Tokens → New token** (read permission is enough)
3. Paste your token below

In [ ]:
from huggingface_hub import login
login(token="hf_your_token_here", add_to_git_credential=False)

## Load Model

`tipsv2-b14-dpt` is the Base/14 DPT variant. Larger variants (`l14-dpt`,
`so400m14-dpt`, `g14-dpt`) give better results at the cost of GPU memory.

`load_model` downloads the DPT model, extracts the backbone, and returns a
`TIPSv2State` with `vision_encoder`, `text_encoder`, `tokenizer`, and `dpt`
— all on the same device.

In [ ]:
MODEL_ID   = "google/tipsv2-b14-dpt"   # swap to l14-dpt / so400m14-dpt for higher quality
RESOLUTION = DEFAULT_RESOLUTION        # 448 px → 32×32 patch grid

state = load_model(MODEL_ID)

---
## Part 1 — PCA & K-Means Feature Visualisation

TIPSv2's vision encoder maps each 14×14-pixel patch to a 768-dim vector.
These vectors encode rich semantic and geometric structure — we can inspect
them without any labels by projecting to 3 dimensions (PCA → RGB) or by
clustering them (K-means).

**What the colours mean in PCA→RGB:**
- The 3 principal components capture the directions of maximum variance
  in the patch feature space.
- Patches with the same semantic content get similar colours across images.
- Surfaces, edges, and background tend to separate into distinct hues.

### 1a — Fruits360 Reference Images

In [ ]:
GALLERY_CLASS = "Apple Red"   # prefix match — change to any Fruits360 class
N_REF         = 6

In [ ]:
ref_class = find_class(GALLERY_CLASS, classes)
ref_paths = get_image_paths(TRAIN_ROOT, ref_class, N_REF)
print(f"Class: {ref_class}  |  {len(ref_paths)} images")

show_pca_gallery(
    ref_paths, state, resolution=RESOLUTION,
    class_name=ref_class,
    save_path="tipsv2_pca_gallery.png",
)

### 1b — Scene Image

The first PCA component often correlates with scene depth because nearby
objects share different statistical properties (texture scale, focus) compared
to distant ones — even though TIPSv2 was never trained with depth supervision.

In [ ]:
show_pca_panel(
    SCENE_PATH, state, resolution=RESOLUTION,
    title=SCENE_PATH.name,
    save_path="tipsv2_pca_scene.png",
)

### 1c — K-Means Feature Clustering

K-means groups patches by feature similarity without any text supervision.
With a small k (3–5) the clusters tend to recover coarse object boundaries;
larger k values reveal finer texture and lighting regions.

In [ ]:
show_kmeans_sweep(
    SCENE_PATH, state,
    cluster_counts=(3, 5, 8, 12),
    resolution=RESOLUTION,
    save_path="tipsv2_kmeans_sweep.png",
)

---
## Part 2 — Zero-Shot Text-Guided Segmentation

Because TIPSv2 patch tokens and text embeddings share the same 768-dim space,
a list of class names can serve directly as prototypes — no reference images,
no bounding boxes, no training.

**Pipeline:**
```
class names
    │  encode_text_classes()  — 9 TCL templates, averaged
    ▼
(N, 768) class embeddings
    │
    │  cosine similarity with every scene patch
    ▼
(sp, sp, N) similarity map  →  upsample  →  argmax  →  label map
```

**TCL template ensemble:** instead of using the raw class name ("apple"),
9 different phrasings are encoded and averaged. This reduces sensitivity
to prompt phrasing — the same trick used in CLIP zero-shot evaluation.

**Value-attention features:** we extract features from only the value (V)
stream of the last ViT block, skipping the query/key attention mixing.
This preserves sharper spatial boundaries at the cost of a small overhead.

### 2a — Fruit Scene Segmentation

In [ ]:
# Classes present in the scene — add or remove as needed
FRUIT_CLASSES = ["apple", "pear", "background"]

In [ ]:
show_zeroseg(
    SCENE_PATH, FRUIT_CLASSES, state,
    use_value_attention=True,
    resolution=RESOLUTION,
    save_path="tipsv2_zeroseg_fruits.png",
)

### 2b — Standard vs Value-Attention Features

The value-attention trick (Part 2 pipeline above) typically produces crisper
boundaries. This cell runs both and places them side by side.

In [ ]:
show_zeroseg_comparison(
    SCENE_PATH, FRUIT_CLASSES, state,
    resolution=RESOLUTION,
    save_path="tipsv2_zeroseg_comparison.png",
)

### 2c — Custom Class Experiment

Because TIPSv2 is a vision-language model with broad concept coverage, any
English noun phrase works as a class name. Try adding classes like `"stem"`,
`"shadow"`, `"label"`, or `"wooden_surface"` to push the boundaries.

In [ ]:
CUSTOM_CLASSES = [] # TODO

show_zeroseg(
    SCENE_PATH, CUSTOM_CLASSES, state,
    use_value_attention=True,
    resolution=RESOLUTION,
    save_path="tipsv2_zeroseg_custom.png",
)

---
## Part 3 — Monocular Depth & Surface Normals (DPT)

The `-dpt` variant adds a **DPT (Dense Prediction Transformer)** head on top
of the **frozen** TIPS backbone. It was trained on the **NYU Depth V2** dataset
to predict:

- **Depth** — distance from the camera for every pixel (warm = near, cool = far)
- **Surface normals** — the 3-D orientation of each surface patch, encoded as
  (X, Y, Z) mapped to (R, G, B)

Key insight: the TIPS backbone was **never trained with depth supervision** — all
depth information emerges from the rich spatial features learned via contrastive
text-image training. The DPT head simply reads this geometry out of the frozen
features.

In [ ]:
show_depth_normals(
    SCENE_PATH, state,
    resolution=RESOLUTION,
    save_path="tipsv2_depth_normals.png",
)

### 3b — Depth on a Fruits360 Reference Image

Fruits360 images have a white background and a single fruit. The depth map
should assign different values to the fruit surface vs the background,
and the normals map should recover the curvature of the fruit.

In [ ]:
# Use the first reference image from the gallery
show_depth_normals(
    ref_paths[0], state,
    resolution=RESOLUTION,
    save_path="tipsv2_depth_normals_fruit.png",
)

---
## Part 4 — Supervised Segmentation (ADE20K, DPT)

A second DPT head was trained on **ADE20K** — 150 semantic categories covering
indoor and outdoor scenes (walls, floors, furniture, vegetation, vehicles, …).

Unlike the zero-shot approach in Part 2, this head:
- Uses **learned** class boundaries (not cosine similarity thresholding)
- Outputs **per-pixel logits** for all 150 classes simultaneously
- Produces sharper boundaries because the head is task-specifically trained

The ADE20K vocabulary does not contain fruit-specific classes, so the model
will map fruits to the nearest available concept (`food`, `plate`, `ball`, …).
This is expected and informative — it shows the domain gap between ADE20K and
the Fruits360 scene.

In [ ]:
show_supervised_seg(
    SCENE_PATH, state,
    resolution=RESOLUTION,
    save_path="tipsv2_supervised_seg.png",
)

---
## Summary

| Capability | What drives it | Supervision |
|---|---|---|
| PCA feature colours | Vision encoder patch tokens | None (unsupervised) |
| K-means clustering | L2-normalised patch tokens | None (unsupervised) |
| Zero-shot segmentation | Patch–text cosine similarity + TCL templates | None (zero-shot) |
| Monocular depth & normals | DPT head on frozen backbone | NYU Depth V2 |
| Semantic segmentation | DPT head on frozen backbone | ADE20K 150 classes |

All five capabilities share the **same frozen TIPS vision encoder** — the rich
spatial features learned via contrastive text-image training generalise to
depth, geometry, and semantic understanding without any task-specific
backbone fine-tuning.

### Where SAM3 is still needed

| Task | TIPSv2 | SAM3 |
|---|---|---|
| **Instance separation** (two touching apples) | ✗ — one region per class | ✓ — distinct mask per object |
| **Pixel-accurate contours** | Patch-grid limited (~14 px) | Sub-pixel (learned decoder) |
| **Calibrated mask confidence** | Raw cosine score | Learned IoU score |
| **Depth & normals** | ✓ (DPT head) | ✗ |
| **Supervised segmentation** | ✓ (ADE20K DPT head) | ✗ |